# FANet on Kvasir-SEG (Kaggle)

Before running: **Settings → Accelerator → GPU T4 x1**, and **Settings → Internet → On** (needs a phone-verified Kaggle account).
Run the cells top to bottom. Training takes roughly 1–3 hours; everything is saved in `/kaggle/working/runs/` and appears under *Output* after **Save Version → Save & Run All**.

## 1. Get the code
The repo is private, so Kaggle needs a GitHub token: create a fine-grained token (read-only, this repo only) at github.com → Settings → Developer settings, then in Kaggle **Add-ons → Secrets** add it as `GITHUB_TOKEN` and tick it for this notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
BRANCH = "claude/nifty-bardeen-kbns7q"   # change to "main" once this branch is merged
!git clone -q -b {BRANCH} https://{token}@github.com/Chaitanya-666/DLproject.git /kaggle/working/dlproject
%cd /kaggle/working/dlproject
!git remote set-url origin https://github.com/Chaitanya-666/DLproject.git   # drop the token from the saved config
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get Kvasir-SEG
Option A downloads it from the official site. Option B: if you attached a Kaggle copy through *Add Data*, set `KAGGLE_COPY` to its folder (the one that contains `images/` and `masks/`).

In [ ]:
import os
KAGGLE_COPY = None   # e.g. "/kaggle/input/kvasir-seg/Kvasir-SEG"
DATA = "/kaggle/working/data/Kvasir-SEG"
if not os.path.exists(DATA):
    if KAGGLE_COPY:
        !mkdir -p /kaggle/working/data && cp -r "{KAGGLE_COPY}" {DATA}
    else:
        !mkdir -p /kaggle/working/data && cd /kaggle/working/data && wget -q https://datasets.simula.no/downloads/kvasir-seg.zip && unzip -q kvasir-seg.zip
!ls {DATA} && ls {DATA}/images | wc -l

## 3. Train/test split (880 / 120, fixed seed)

In [ ]:
!python -m src.split --data {DATA} --test 120

## 4. Quick sanity run (2 minutes)
Catches mistakes before the long run. Dice should already move above zero.

In [ ]:
!python -m src.train --data {DATA} --out runs/sanity --epochs 2 --max-batches 20

## 5. Full training
Official code: batch 2, lr 1e-4, up to 500 epochs. We use batch 8 for speed and stop early after 40 epochs without improvement.

In [ ]:
!python -m src.train --data {DATA} --out runs/kvasir --epochs 200 --batch 8 --lr 1e-4

## 6. Learning curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("runs/kvasir/log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
log.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title="Loss (0.5 BCE + 0.5 Dice)")
log.plot(x="epoch", y=["train_dice", "val_dice"], ax=ax[1], title="Dice")
plt.savefig("runs/kvasir/curves.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Test with iterative refinement (10 rounds, as in the official test script)

In [ ]:
!python -m src.evaluate --ckpt runs/kvasir/best.pt --data {DATA} --split test --iters 10
pd.read_csv("runs/kvasir/eval_test/results.csv").round(4)

Example strips: image | ground truth | Otsu start | round 1 | round 2 | ...

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("runs/kvasir/eval_test/strips/*.png"))[:6]:
    display(Image(p))

## 8. Own dataset: generate, fine-tune, evaluate
Synthetic leaf spots made by script (exact masks, no annotation). For real photos, annotate in Labelme on your laptop, convert with `src.labelme_to_masks`, commit them to `dataset/custom/leaf`, and use that folder instead.

In [ ]:
CUSTOM = "dataset/custom/synthetic_leaf"
if not os.path.exists(CUSTOM + "/images"):
    !python -m src.make_synthetic procedural --out {CUSTOM} --n 60
!python -m src.split --data {CUSTOM} --test 0.2
# Before fine-tuning: how does the polyp model do on leaves?
!python -m src.evaluate --ckpt runs/kvasir/best.pt --data {CUSTOM} --split test --iters 3 --out runs/leaf_before
# Fine-tune from the Kvasir weights
!python -m src.train --data {CUSTOM} --out runs/leaf --init runs/kvasir/best.pt --epochs 60 --lr 5e-5
!python -m src.evaluate --ckpt runs/leaf/best.pt --data {CUSTOM} --split test --iters 3

## 9. Bundle results for download

In [ ]:
!cd runs && zip -qr /kaggle/working/results.zip . -x "*.pt" && ls -lh /kaggle/working/results.zip